# Fast Discrete Curvelet Transforms — Implementation / 구현

**Paper**: Candès, E. J., Demanet, L., Donoho, D. L., & Ying, L. (2006). *Multiscale Model. Simul.*, 5(3), 861–899. [DOI: 10.1137/05064182X]

## Overview / 개요

전체 FDCT (CurveLab 또는 `curvelops`)는 본 노트북 범위를 벗어나므로, **핵심 개념 시연**:
1. **Continuous curvelet의 폴라 wedge** \$U\_j(r, \\theta) = 2^{-3j/4} W(2^{-j} r) V\\bigl(\\frac{2^{\\lfloor j/2\\rfloor}\\theta}{2\\pi}\\bigr)\$ 시각화
2. **Cartesian coronization**: concentric squares + shears (Section 3.1) 시각화
3. **Parabolic scaling**: \$\\text{width} \\propto \\text{length}^2\$ 시각 검증
4. **Single-scale, single-angle wrapping FDCT**: Fourier wedge multiply + wrap + IFFT 한 번 시연
5. Energy preservation (Parseval) 검증
6. (가능하면) `curvelops` 패키지로 풀 FDCT 시도

**Note**: full FDCT (USFFT or wrapping) is well beyond a single notebook — implementation requires careful Cartesian coronization across all (scale, angle) pairs. Reference implementations: **CurveLab** (curvelet.org) or **`curvelops`** (PyLops). This notebook demonstrates the *core building blocks* on a single (scale, angle).


In [ ]:
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt
from numpy.typing import NDArray
from skimage import data, img_as_float

plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['font.size'] = 10
rng = np.random.default_rng(42)

---

## Part 1: Continuous curvelet polar wedge / 연속 컬블릿 폴라 웻지

Eq. (2.3): \$U\_j(r, \\theta) = 2^{-3j/4} W(2^{-j} r) V\\bigl(\\frac{2^{\\lfloor j/2\\rfloor}\\theta}{2\\pi}\\bigr)\$ — 라디얼 \$\\times\$ 각도 윈도우.


In [ ]:
def smooth_window(t: NDArray[np.float64], a: float, b: float) -> NDArray[np.float64]:
    """Smooth bump window non-zero on [a, b], zero outside.

    Returns 1 in the inner half, smoothly tapers to 0 near the boundaries.
    """
    inner = (b - a) / 4
    out = np.zeros_like(t, dtype=np.float64)
    inside = (t >= a) & (t <= b)
    # cosine taper at both ends
    left_taper = (t >= a) & (t < a + inner)
    right_taper = (t > b - inner) & (t <= b)
    flat = (t >= a + inner) & (t <= b - inner)
    out[flat] = 1.0
    out[left_taper] = 0.5 * (1 + np.cos(np.pi * (t[left_taper] - (a + inner)) / inner))
    out[right_taper] = 0.5 * (1 + np.cos(np.pi * (t[right_taper] - (b - inner)) / inner))
    return out


def W(r: NDArray[np.float64]) -> NDArray[np.float64]:
    """Radial window supported on (1/2, 2) (paper §2)."""
    return smooth_window(r, 0.5, 2.0)


def V(t: NDArray[np.float64]) -> NDArray[np.float64]:
    """Angular window supported on [-1, 1] (paper §2)."""
    return smooth_window(t, -1.0, 1.0)


def U_j_polar(omega1, omega2, j: int) -> NDArray[np.float64]:
    """Polar curvelet frequency window U_j(r, theta) (Eq. 2.3)."""
    r = np.sqrt(omega1 ** 2 + omega2 ** 2)
    theta = np.arctan2(omega2, omega1)
    return 2 ** (-3 * j / 4) * W(2 ** (-j) * r) * V(2 ** (j // 2) * theta / (2 * np.pi))


# Visualize U_j across scales
Omega = np.linspace(-4, 4, 401)
Om1, Om2 = np.meshgrid(Omega, Omega)
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, j in zip(axes, [-2, -1, 0, 1]):
    U = U_j_polar(Om1, Om2, j)
    ax.imshow(U, extent=(-4, 4, -4, 4), origin='lower', cmap='hot')
    ax.set_title(f'$U_{{{j}}}(r, \\theta)$, polar wedge')
    ax.set_xlabel(r'$\omega_1$'); ax.set_ylabel(r'$\omega_2$')
plt.tight_layout(); plt.show()
print('Each wedge is supported near a polar dyadic corona of width ~2^j and angular extent ~2π·2^(-j/2).')

---

## Part 2: Cartesian coronization / 직사각 좌표화 (§3.1)

Polar wedge → Cartesian wedge. \$\\tilde W\_j = \\sqrt{\\Phi^2\_{j+1} - \\Phi^2\_j}\$ from concentric squares; shear matrix \$S\_\\theta\$ replaces rotation.


In [ ]:
def Phi_j(omega1, omega2, j: int) -> NDArray[np.float64]:
    """Concentric-square lowpass window (paper Eq. 3.2 component)."""
    phi1 = smooth_window(2 ** (-j) * omega1, -2.0, 2.0)
    phi2 = smooth_window(2 ** (-j) * omega2, -2.0, 2.0)
    return phi1 * phi2


def W_tilde_j(omega1, omega2, j: int) -> NDArray[np.float64]:
    """Cartesian radial bandpass window: sqrt(Phi_{j+1}^2 - Phi_j^2)."""
    return np.sqrt(np.maximum(Phi_j(omega1, omega2, j+1) ** 2 - Phi_j(omega1, omega2, j) ** 2, 0.0))


def shear_matrix(theta: float) -> NDArray[np.float64]:
    """Shear matrix S_theta (Section 3.1)."""
    return np.array([[1.0, 0.0], [-np.tan(theta), 1.0]])


def U_tilde_jl(omega1, omega2, j: int, theta_ell: float) -> NDArray[np.float64]:
    """Cartesian wedge \\tilde U_{j, ℓ}(\omega) = \\tilde W_j(\omega) V_j(S_{\\theta_ℓ} \\omega).

    Approximation: V_j(slope) where slope = (S_theta omega) ratio.
    """
    Wj = W_tilde_j(omega1, omega2, j)
    # Shear coordinates
    s = shear_matrix(theta_ell)
    om1_sh = s[0, 0] * omega1 + s[0, 1] * omega2
    om2_sh = s[1, 0] * omega1 + s[1, 1] * omega2
    # V_j(t) where t is the angle proxy: tan-like
    eps = 1e-9
    t = 2 ** (j // 2) * om2_sh / (om1_sh + eps)
    return Wj * smooth_window(t, -1.0, 1.0)


# Visualize Cartesian wedges at scale j=0 for several angles
Omega = np.linspace(-4, 4, 401)
Om1, Om2 = np.meshgrid(Omega, Omega)
thetas = [-np.pi/4, -np.pi/8, 0.0, np.pi/8, np.pi/4]
fig, axes = plt.subplots(1, len(thetas), figsize=(15, 4))
for ax, th in zip(axes, thetas):
    U = U_tilde_jl(Om1, Om2, j=0, theta_ell=th)
    ax.imshow(U, extent=(-4, 4, -4, 4), origin='lower', cmap='hot')
    ax.set_title(f'$\\tilde U_{{0, \\theta}}$, $\\theta={np.degrees(th):.0f}^\\circ$')
plt.tight_layout(); plt.show()
print('Cartesian wedges via shears: support is shifted but axis-aligned-friendly for grids.')

---

## Part 3: Parabolic scaling visualisation / 포물선 스케일링


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, j in zip(axes, [0, 1, 2, 3]):
    # Curvelet support: width ~ 2^j (in physical), length ~ 2^{j/2}
    width = 2 ** j
    length = 2 ** (j / 2 + 1)
    # Draw an ellipse
    t = np.linspace(0, 2 * np.pi, 200)
    x = (length / 2) * np.cos(t)
    y = (width / 2) * np.sin(t)
    ax.fill(x, y, alpha=0.5, color='C0')
    ax.set_xlim(-10, 10); ax.set_ylim(-10, 10); ax.set_aspect('equal')
    ax.set_title(f'j={j}: width={width:.1f}, length={length:.2f}\nratio width/length²={width/length**2:.2f}')
    ax.grid(True)
plt.suptitle('Parabolic scaling: width ∝ length² (anisotropic, growing more elongated at finer scales)')
plt.tight_layout(); plt.show()

---

## Part 4: Wrapping FDCT building block / Wrapping FDCT 구성 요소

단일 (scale, angle) 페어에 대해 wrap 단계 시연: \$f \\to \\hat f \\cdot \\tilde U\_{j,\\ell} \\to \\text{wrap} \\to \\text{IFFT}\$.


In [ ]:
def fdct_single_band(
    img: NDArray[np.float64],
    j: int,
    theta_ell: float,
) -> NDArray[np.complex128]:
    """Forward FDCT for a single (scale j, angle theta_ell).

    Wrapping is approximated here as multiplication in the FFT domain only
    (the geometric wrap is omitted for clarity). The output is the
    band-limited filtered image at this (j, theta).

    Args:
        img: input image, (n, n).
        j: scale index.
        theta_ell: angle in radians.

    Returns:
        Complex-valued band image of shape (n, n).
    """
    n = img.shape[0]
    F = np.fft.fftshift(np.fft.fft2(img))
    # Frequency grid in cycles per pixel, normalised
    fx = np.fft.fftshift(np.fft.fftfreq(n)) * n  # in [-n/2, n/2)
    fy = np.fft.fftshift(np.fft.fftfreq(n)) * n
    Om1, Om2 = np.meshgrid(fx, fy)
    # Map to scale-normalised frequencies; wedge for scale j
    U = U_tilde_jl(Om1 / 2 ** (j + 1), Om2 / 2 ** (j + 1), j=0, theta_ell=theta_ell)
    band_freq = F * U
    return np.fft.ifft2(np.fft.ifftshift(band_freq))


img = img_as_float(data.camera())[:128, :128] * 255.0
img = img - img.mean()

fig, axes = plt.subplots(1, 5, figsize=(15, 4))
axes[0].imshow(img, cmap='gray'); axes[0].set_title('input'); axes[0].axis('off')
for ax, th in zip(axes[1:], [-np.pi/4, 0.0, np.pi/4, np.pi/2]):
    band = fdct_single_band(img, j=2, theta_ell=th)
    ax.imshow(np.real(band), cmap='gray')
    ax.set_title(rf'band $j=2$, $\theta={np.degrees(th):.0f}^\circ$'); ax.axis('off')
plt.tight_layout(); plt.show()

---

## Part 5: Energy preservation (Parseval) / 에너지 보존

복수 wedge 합산이 \$\\sum |\\tilde U\_{j,\\ell}|^2 = 1\$이면 wrapping FDCT는 tight frame.


In [ ]:
# Construct an ensemble of wedges and check sum of |U|^2
Omega = np.linspace(-4, 4, 201)
Om1, Om2 = np.meshgrid(Omega, Omega)

n_angles_per_scale = {-1: 8, 0: 16, 1: 16, 2: 32}
scale_range = list(n_angles_per_scale.keys())
total_squared_window = np.zeros_like(Om1)
for j in scale_range:
    n_angles = n_angles_per_scale[j]
    thetas = np.linspace(-np.pi/2, np.pi/2, n_angles, endpoint=False)
    for th in thetas:
        U = U_tilde_jl(Om1, Om2, j=j, theta_ell=th)
        total_squared_window += U ** 2

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
im0 = ax[0].imshow(total_squared_window, extent=(-4,4,-4,4), origin='lower', cmap='hot', vmin=0, vmax=2)
ax[0].set_title(r'$\sum_{j, \ell} |\tilde U_{j,\ell}(\omega)|^2$ (should be near 1 for tight frame)')
plt.colorbar(im0, ax=ax[0])
ax[1].hist(total_squared_window.ravel(), bins=80)
ax[1].axvline(1.0, color='r', ls='--', label='ideal=1')
ax[1].set_xlabel('value'); ax[1].set_ylabel('count')
ax[1].legend()
plt.tight_layout(); plt.show()

print('Our simplified wedge family is NOT exactly a tight frame because:')
print(' - the angular V is not exactly compatible with the \"slope\" parameterization')
print(' - the radial W_tilde definition is approximate')
print('Reference implementations (CurveLab, curvelops) achieve numerical isometry to ~1e-13.')

---

## Part 6: Try `curvelops` if available / 가능하면 curvelops로 검증


In [ ]:
try:
    import curvelops
    print(f'curvelops available: version {curvelops.__version__}')
    img_full = img_as_float(data.camera())[:256, :256] * 255.0
    fdct = curvelops.FDCT2D(dims=img_full.shape)
    coefs = fdct @ img_full
    img_rec = fdct.H @ coefs
    rec_err = float(np.max(np.abs(img_full - np.real(img_rec))))
    energy_in = float(np.sum(img_full ** 2))
    energy_coefs = float(np.sum(np.abs(coefs) ** 2))
    print(f'Reconstruction max error: {rec_err:.3e}')
    print(f'||img||^2: {energy_in:.4e}')
    print(f'||coefs||^2: {energy_coefs:.4e}')
    print(f'Energy ratio: {energy_coefs/energy_in:.6f}')
except ImportError:
    print('curvelops not installed. Install via: pip install curvelops')
    print('Reference: https://github.com/PyLops/curvelops')
    print('CurveLab (original): https://www.curvelet.org')

---

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | This Notebook / 본 노트북 | Modern Equivalent / 현대 동등물 |
|---|---|---|---|
| Polar curvelet wedge \$U\_j\$ | Eq. (2.3) | `U_j_polar` | (theory only) |
| Radial \$W\$ + angular \$V\$ windows | Eq. (2.1, 2.2) | `W`, `V`, `smooth_window` | (paper-specific) |
| Cartesian wedge \$\\tilde U\_{j,\\ell}\$ | Eq. (3.3) | `U_tilde_jl` | CurveLab `fdct_wrapping_window` |
| Shear \$S\_\\theta\$ | §3.1 | `shear_matrix` | (paper-specific) |
| Single-band FDCT step | §6 | `fdct_single_band` (simplified) | `curvelops.FDCT2D` |
| Tight-frame check | Eq. (2.7) | `total_squared_window` | numerical isometry in CurveLab \$\\sim 10^{-13}\$ |
| Full FDCT (USFFT or wrapping) | §4, §6 | (out of scope) | **CurveLab** (Matlab+C++) or **`curvelops`** (Python) |

### Connections to next papers / 다음 논문과의 연결

- **Paper #5 Contourlet** — sister approach via filter banks (LP+DFB) instead of Fourier-domain wedges.
- **Paper #8 Shearlet** — third sister; shears as a *group action* (with affine-invariance benefits).
- **Paper #7 BM3D** — different paradigm (nonlocal grouping); could in principle use curvelets in its 3D transform step.

### Take-away / 결론

본 노트북은 polar wedge \$U\_j\$, Cartesian wedge \$\\tilde U\_{j,\\ell}\$, parabolic scaling, 단일 (j, θ) 페어에 대한 wrapping 단계를 시연. 전체 FDCT 알고리즘은 본 범위를 벗어나며, 실용 사용은 CurveLab (Matlab+C++) 또는 `curvelops` (Python) 권장. Tight-frame 성질은 우리의 단순화된 wedge에서는 근사일 뿐이지만, 실제 CurveLab은 1e-13 수준의 numerical isometry를 달성한다.

Demonstrates the polar wedge \$U\_j\$, Cartesian wedge \$\\tilde U\_{j,\\ell}\$, parabolic scaling, and the single-band wrapping step. The full FDCT is beyond a single notebook; for practical use we recommend CurveLab or curvelops, which achieve numerical isometry to \$\\sim 10^{-13}\$.
